# Pandas 第6课：进阶专题与 2.x 新特性

> 前5课覆盖了建表、读写选择、清洗、分组聚合合并、综合练习。这一课补三块前面完全没碰的东西：
> 1. 写法层面：链式调用、query/eval、命名聚合——同样的逻辑，写得更专业
> 2. 结构层面：MultiIndex、时间序列resample/rolling、reshape四兄弟（pivot/melt/stack/explode）——之前没接触过的数据形态
> 3. 底层层面：pandas 2.x 的 Copy-on-Write 机制、categorical内存优化、pd.NA可空类型——解释你在练习卷里已经遇到过但没细讲的现象（比如'单价缺失后整列变成float'）
>
> 参考：pandas官方文档、Jake VanderPlas《Python Data Science Handbook》、Matt Harrison《Effective Pandas》、pandas PDEP-0007提案。

In [1]:
import pandas as pd
import numpy as np
np.random.seed(7)
pd.set_option('display.max_columns', 20)

n = 40
df = pd.DataFrame({
    '订单号': range(2001, 2001+n),
    '日期': pd.date_range('2024-01-01', periods=n, freq='D'),
    '客户': np.random.choice(['张三','李四','王五','赵六','钱七'], n),
    '城市': np.random.choice(['昆明','北京','上海','广州','成都'], n),
    '品类': np.random.choice(['数码','服饰','食品','图书'], n),
    '单价': np.random.randint(20, 500, n).astype(float),
    '数量': np.random.randint(1, 6, n),
})
df.loc[[3, 15, 27], '单价'] = np.nan   # 埋几个缺失，后面讲pd.NA会用到
df['金额'] = df['单价'] * df['数量']
print(df.shape)
df.head()

(40, 8)


,订单号,日期,客户,城市,品类,单价,数量,金额
0,2001,2024-01-01,钱七,成都,图书,302.0,4,1208.0
1,2002,2024-01-02,李四,上海,图书,340.0,5,1700.0
2,2003,2024-01-03,赵六,北京,图书,272.0,1,272.0
3,2004,2024-01-04,赵六,广州,数码,NaN,4,NaN
4,2005,2024-01-05,钱七,昆明,数码,234.0,2,468.0


## 1. 方法链式调用（method chaining）与 `.pipe()`

pandas几乎所有方法都返回**新对象**而不就地修改（`inplace=True`早就不推荐用了，2.x里配合Copy-on-Write甚至可能不生效，第6节会讲原因）。这个设计天然支持"链式"写法。

链式的价值不是炫技，是**消除没有语义的中间变量**。`df2`、`df3`、`tmp`这些名字不会告诉你"这一步做了什么"，而链式代码从上到下读就是一段数据变换的叙事：

```python
result = (
    df.query("城市 == '昆明'")          # 第1步：筛选
      .assign(单价折后=lambda d: d['单价']*0.9)  # 第2步：造列
      .groupby('品类')
      .agg(总额=('金额','sum'))          # 第3步：聚合
)
```

`.pipe(func)` 用来把**不是DataFrame自带方法**的自定义函数也接进链条里，避免链条被一句`df = my_func(df)`截断。

**诚实地说链式的缺点**：调试不方便——链条中间任意一步出错，不能直接在那一行加`print()`。实践建议：开发阶段先拆成变量方便打断点，逻辑稳定后再合并成链。

来源：towardsdatascience "This One Pandas Pattern Separates Beginners From Intermediate Users"；Matt Harrison《Effective Pandas》。

In [2]:
def add_discount(data, rate=0.9):
    return data.assign(折后单价=(data['单价'] * rate).round(1))  # 自定义函数，用pipe接进链条

result = (
    df.dropna(subset=['单价'])                      # 第1步：去掉单价缺失的行
      .query("城市 in ['昆明','北京']")               # 第2步：筛选城市
      .pipe(add_discount, rate=0.85)                # 第3步：接入自定义函数
      .groupby('品类')
      .agg(订单数=('订单号','count'), 均价=('折后单价','mean'))
      .round(1)
      .sort_values('均价', ascending=False)          # 第4步：排序
)
result

,订单数,均价
品类,,
食品,1,393.6
图书,4,312.4
服饰,3,219.8
数码,4,176.6


## 2. `query()` 和 `eval()`：可读性之外，性能故事更有意思

`query()`/`eval()` 底层如果装了 `numexpr` 引擎，会把表达式编译成接近C速度的向量化计算，对**大表、复杂布尔/算术表达式**有实测加速。但对小表或简单条件，调用 `numexpr` 本身的开销反而比普通布尔索引更慢——这是"性能优化不能无脑套用"的好例子，实测比瞎猜靠谱。

可读性上的差别很直接：

```python
df.query("城市=='昆明' and 金额>1000")
# 等价于
df[(df['城市']=='昆明') & (df['金额']>1000)]
```

局限：列名有空格/特殊符号要用反引号包裹；不支持所有Python表达式（比如某些链式方法调用没法直接塞进字符串里）。

来源：Jake VanderPlas《Python Data Science Handbook》"High-Performance Pandas: eval() and query()"章节；pandas官方文档。

In [3]:
# 造一张大表，才能看出query的性能差异（小表两者差不多，甚至query更慢）
big = pd.DataFrame(np.random.rand(1_000_000, 3), columns=['a','b','c'])

import time
t0 = time.time()
r1 = big[(big['a'] > 0.3) & (big['b'] < 0.7)]
t1 = time.time()
r2 = big.query("a > 0.3 and b < 0.7")
t2 = time.time()
print(f"布尔索引耗时: {t1-t0:.4f}s")
print(f"query耗时:   {t2-t1:.4f}s")
print("两种写法结果是否一致:", len(r1) == len(r2))

布尔索引耗时: 0.0131s
query耗时:   0.0236s
两种写法结果是否一致: True


## 3. MultiIndex（多级索引）——你已经用过，只是没有名字

`groupby(['A','B'])` 默认就会产生 MultiIndex 结果，第4课/练习卷里你已经在用了。理解它的关键：**MultiIndex不是新数据结构**，只是Index对象的标签从单个值变成了元组。

**核心操作三件套：**
- 创建：`set_index(['col1','col2'])` 或 `groupby` 自动产生
- 按层取值：`.xs(key, level='城市')` 比手写 `.loc[(a,b),:]` 直观
- 跨层切片：`.loc[(slice(None), '某值'), :]`

**`stack`/`unstack` 和你已经会的 `pivot_table` 是同一族操作：**
- `unstack`：把行索引的某一层"掀"到列上（变宽）
- `stack`：反向操作（变长）
- `pivot_table` 其实是 `groupby + unstack` 的封装——理解了这一点，三个方法就不再是三个孤立的知识点

来源：Kanaries "Hierarchical Indexing Guide"；pandas官方MultiIndex文档；StackOverflow "why and when should use stack()/unstack()"。

In [4]:
df2 = df.dropna(subset=['单价']).copy()
df2['月份'] = df2['日期'].dt.month

# groupby多个key，默认产生MultiIndex
grouped = df2.groupby(['城市','月份'])['金额'].sum()
print("grouped的索引类型:", type(grouped.index))
print(grouped.head(6))

# xs：按某一层取值，比 .loc[(a,slice(None))] 直观
print("\n只看昆明各月:")
print(grouped.xs('昆明', level='城市'))

grouped的索引类型: <class 'pandas.core.indexes.multi.MultiIndex'>
城市  月份
上海  1     4894.0
北京  1     6258.0
    2     1266.0
广州  1     7691.0
    2     1233.0
成都  1     3233.0
Name: 金额, dtype: float64

只看昆明各月:
月份
1    1618.0
Name: 金额, dtype: float64


In [5]:
# unstack：把"月份"这一层从行索引掀到列上，变成宽表
wide_raw = grouped.unstack('月份')          # 不填0，保留真实缺失（有的城市某月没有订单）
print("unstack后（行=城市，列=月份，NaN表示当月无订单):")
print(wide_raw)

# stack：反向操作，变回长表——验证是否还原（必须用没填过0的版本，否则会凭空多出被填的组合）
back = wide_raw.stack()
print("\nstack还原后和原grouped是否一致:", grouped.sort_index().equals(back.sort_index()))

# 展示用可以填0，但填0之后就不再是"原始形态"，不能拿来验证还原
wide = wide_raw.fillna(0)
print("\n填0后用于展示（行=城市，列=月份):")
print(wide)

# 对比：用pivot_table直接生成同样的结果，验证"pivot_table = groupby+unstack"
pt = df2.pivot_table(index='城市', columns='月份', values='金额', aggfunc='sum', fill_value=0)
print("\npivot_table结果是否与groupby+unstack(填0后)一致:", wide.equals(pt))

unstack后（行=城市，列=月份，NaN表示当月无订单):
月份       1       2
城市                
上海  4894.0     NaN
北京  6258.0  1266.0
广州  7691.0  1233.0
成都  3233.0  2994.0
昆明  1618.0     NaN

stack还原后和原grouped是否一致: True

填0后用于展示（行=城市，列=月份):
月份       1       2
城市                
上海  4894.0     0.0
北京  6258.0  1266.0
广州  7691.0  1233.0
成都  3233.0  2994.0
昆明  1618.0     0.0

pivot_table结果是否与groupby+unstack(填0后)一致: True


## 4. 时间序列：resample、rolling/expanding —— 和"聚合vs transform"是同一个思维模型

**先对比一下你已经会的东西**：`groupby`聚合后行数变少（一组一个值），`transform`后行数不变（结果广播回每行）。

时间序列版本完全对应：
- `resample`：**改变数据点数量**（按天数据聚合成按月，行数变少）—— 对应"聚合"
- `rolling(window)`：**行数不变**，只是给每行配一个滑动窗口内的统计值 —— 对应"transform"
- `expanding()`：窗口从头开始不断扩大到当前行，做累计统计（比如累计销售额，KPI仪表盘常见）

来源：pandas官方`resample`文档；Dataquest "Time Series Analysis with Pandas"。

In [6]:
daily = df2.set_index('日期')['金额'].resample('D').sum()  # 按天补全（没有订单的天是0/NaN）
print("按天序列长度:", len(daily))

weekly = daily.resample('W').sum()   # resample：行数变少，聚合到"周"粒度
print("\n按周聚合后长度:", len(weekly))
print(weekly.head())

rolling7 = daily.rolling(window=7).mean()   # rolling：行数不变，7日滚动均值
print("\nrolling结果长度是否等于原序列:", len(rolling7) == len(daily))

cum = daily.expanding().sum()   # expanding：累计求和，典型的"月度/年度累计KPI"
print("\n最后5天的累计销售额:")
print(cum.tail())

按天序列长度: 40

按周聚合后长度: 6
日期
2024-01-07    5038.0
2024-01-14    7055.0
2024-01-21    4756.0
2024-01-28    4397.0
2024-02-04    4063.0
Freq: W-SUN, Name: 金额, dtype: float64

rolling结果长度是否等于原序列: True

最后5天的累计销售额:
日期
2024-02-05    26101.0
2024-02-06    27081.0
2024-02-07    28548.0
2024-02-08    28653.0
2024-02-09    29187.0
Freq: D, Name: 金额, dtype: float64


## 5. categorical dtype：内存优化的原理，不只是"变快"

`object`类型存字符串列，**每行都各自持有一份完整字符串**（指针+堆内存）。`category`类型只存"类别表"一份，每行只存一个小整数编码——当一列的取值种类远小于行数时（比如你数据里的"城市""品类"，正是这种典型场景），内存能压到几分之一。

**什么场景该用**：低基数（unique值少）的字符串列，且会频繁做筛选/groupby。**不要用在**高基数列（比如订单号、姓名，如果几乎每行都不同，编码表反而多此一举）。

来源：towardsdatascience "Staying sane while adopting pandas categorical datatypes"；pandas官方内存文档。

In [7]:
# 放大到10万行来看效果（用你练习卷同样的城市/品类结构）
big_city = pd.DataFrame({
    '城市': np.random.choice(['昆明','北京','上海','广州','成都'], 100_000)  # 只有5个取值
})

before = big_city.memory_usage(deep=True).sum()
big_city['城市_分类'] = big_city['城市'].astype('category')
after = big_city[['城市_分类']].memory_usage(deep=True).sum()

print(f"object存储: {before/1024:.1f} KB")
print(f"category存储: {after/1024:.1f} KB")
print(f"压缩到约: {after/before*100:.1f}%")

object存储: 8398.6 KB
category存储: 98.4 KB
压缩到约: 1.2%


## 6. pandas 2.x 的核心变化：这是"跟上最新技术"最该看的一节

### 6.1 为什么整数列一有缺失就变成float？—— 旧pandas的历史包袱

旧pandas用`NaN`表示缺失，而`NaN`**本身就是一个float值**（IEEE 754浮点数的一个特殊值）。NumPy的整数类型（`int64`）没有办法表示"缺失"这个状态，所以一旦一列整数出现缺失，pandas被迫把**整列**升级成float——这正是你在综合练习卷"第14题：单价有缺失"里看到的现象，单价从整数变成了带`.0`的浮点数。

### 6.2 解决方案：pd.NA 与可空扩展类型

pandas引入了`Int64`、`boolean`等**可空扩展类型**（注意首字母大写，区别于numpy原生的`int64`）。这些类型让整数列**带缺失值也能保持整数身份**，缺失值用`pd.NA`表示，不再强制转float。

`pd.NA`比`NaN`更严格：它遵循**三值逻辑**——`True | pd.NA` 结果确定是 `True`（因为无论NA实际是什么，`True|任何值`都是`True`），但`False | pd.NA`结果是`pd.NA`（不确定），这是"逻辑上能确定的就返回确定值，不能确定的就诚实说不知道"。

来源：pandas官方nullable integer/boolean文档。

In [8]:
nums = pd.Series([10, 20, np.nan, 40])
print("旧方式 dtype:", nums.dtype, "  -> 整数被迫变float:", nums.tolist())

nullable = pd.Series([10, 20, pd.NA, 40], dtype='Int64')   # 注意大写Int64
print("\n可空整数类型 dtype:", nullable.dtype, "  -> 仍是整数身份:", nullable.tolist())

# 三值逻辑演示
print("\ntrue | pd.NA  =", True | pd.NA)    # 确定为True，不管NA是什么
print("false | pd.NA =", False | pd.NA)   # 不确定，结果就是NA

旧方式 dtype: float64   -> 整数被迫变float: [10.0, 20.0, nan, 40.0]

可空整数类型 dtype: Int64   -> 仍是整数身份: [10, 20, <NA>, 40]

true | pd.NA  = True
false | pd.NA = <NA>


### 6.3 Copy-on-Write（写时复制）：从根本上解决了 SettingWithCopyWarning 的老问题

**先看老问题**：`df[df.a>1]['b']=x` 是两步独立的操作——先`__getitem__`筛选出一个子集，再对这个子集`__setitem__`赋值。这个子集到底是原数据的**视图**还是**独立副本**，旧pandas自己都不保证（取决于底层数据block的内存布局），于是赋值有时生效、有时不生效，还会弹出`SettingWithCopyWarning`让人心里发慌。

**pandas 2.0引入、未来版本默认开启的Copy-on-Write机制**：让**每一次索引操作在语义上都表现为copy**。这句话的意思是——`df[df.a>1]`现在稳定地是一份独立数据，你在它上面赋值，改的从来不是原始df。这从设计上直接**废除了链式赋值能生效的前提**，所以官方计划在CoW全面落地后彻底移除这个警告，因为"两步赋值能改到原数据"这件事本身已经不存在了。

**正确写法从来只有一种，现在有了更硬的理由**：`.loc[row_mask, 'col'] = x` 是**一步**完成"定位+赋值"，语义上明确是原地修改，不存在"到底改的是谁"的歧义。

来源：pandas官方Copy-on-Write用户指南；PDEP-0007提案原文（最权威，直接说明了移除警告的设计动机）；pandas官方Indexing文档"Returning a view versus a copy"。

In [9]:
import warnings

toy = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})

print("=== 场景1：CoW关闭（pandas 2.3.3的默认行为）===")
pd.set_option('mode.copy_on_write', False)
d1 = toy.copy()
sub = d1[d1['a'] > 1]        # 第一步 __getitem__，是不是view/copy不确定
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    sub['b'] = 999            # 第二步 __setitem__，链式索引赋值
    print("弹出SettingWithCopyWarning:", len(w) > 0)
print("原始d1有没有被改动:\n", d1)

print("\n=== 场景2：CoW开启（未来默认行为）===")
pd.set_option('mode.copy_on_write', True)
d2 = toy.copy()
sub2 = d2[d2['a'] > 1]
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    sub2['b'] = 999
    print("弹出警告:", len(w) > 0, "  <- 没有警告了，因为规则已经确定，不用再提醒你风险")
print("原始d2有没有被改动:\n", d2)

pd.set_option('mode.copy_on_write', False)  # 还原设置，不影响后面的cell

print("\n=== 正确写法：不管CoW开不开，这样写永远没有歧义 ===")
d3 = toy.copy()
d3.loc[d3['a'] > 1, 'b'] = 999    # 一步完成"定位+赋值"
print(d3)

=== 场景1：CoW关闭（pandas 2.3.3的默认行为）===
原始d1有没有被改动:
    a   b
0  1  10
1  2  20
2  3  30

=== 场景2：CoW开启（未来默认行为）===
弹出警告: False   <- 没有警告了，因为规则已经确定，不用再提醒你风险
原始d2有没有被改动:
    a   b
0  1  10
1  2  20
2  3  30

=== 正确写法：不管CoW开不开，这样写永远没有歧义 ===
   a    b
0  1   10
1  2  999
2  3  999


### 6.4 一句话带过：Arrow 后端

`pd.ArrowDtype` 让pandas底层用Apache Arrow内存格式而不是NumPy，对字符串处理和跨语言互操作（比如和Polars、DuckDB共享内存而不用互相转换复制）有性能优势，是"面向未来"的知识点。现阶段不需要深入用，知道"为什么存在"就够了——数据生态在往一个大家能共享同一份内存格式的方向走。

## 7. groupby 进阶：命名聚合 + transform/apply/agg 的性能与语义

**命名聚合（named aggregation）**：你在练习卷第19题用过 `.agg({'金额':'sum', ...})`，这种写法容易产生MultiIndex列名，还要手动拍平。更优写法：

```python
df.groupby('城市').agg(总额=('金额','sum'), 均价=('单价','mean'))
```

直接得到扁平、语义化的列名，不用二次改名。

**transform vs apply vs agg**：
- `agg`：输出行数变少（一组一个值）
- `transform`：输出行数不变（结果广播回每一行）——你在第21题用过
- `apply`：最灵活，但也最慢，因为它对每个分组单独跑一次Python函数、再逐组拼接结果。**能用agg/transform解决的，不要用apply**，这是一条实测验证过的经验规则，不是随口一说。

来源：StackOverflow "Transform vs. aggregate in Pandas groupby"高赞回答；Kanaries "Aggregation, Transform, Apply 2026 Guide"。

In [10]:
# 命名聚合：一次性算出语义化列名
named = df2.groupby('客户').agg(
    订单数=('订单号', 'count'),
    总消费=('金额', 'sum'),
    最大单笔=('金额', 'max')
).round(1)
print(named)

# transform vs apply 性能对比：组内标准分 (x-组均值)/组标准差
big2 = pd.DataFrame({
    'grp': np.random.choice(list('ABCDE'), 200_000),
    'val': np.random.rand(200_000)
})

import time
t0 = time.time()
z_transform = big2.groupby('grp')['val'].transform(lambda x: (x - x.mean()) / x.std())
t1 = time.time()
z_apply = big2.groupby('grp')['val'].apply(lambda x: (x - x.mean()) / x.std()).reset_index(level=0, drop=True)
t2 = time.time()
print(f"\ntransform耗时: {t1-t0:.4f}s")
print(f"apply耗时:     {t2-t1:.4f}s")
print("两种结果是否一致:", np.allclose(z_transform.sort_index(), z_apply.sort_index()))

    订单数      总消费    最大单笔
客户                      
张三   13  10520.0  1488.0
李四    5   5612.0  1700.0
王五    4   2706.0  1314.0
赵六    6   2972.0   990.0
钱七    9   7377.0  2290.0

transform耗时: 0.0223s
apply耗时:     0.0367s


两种结果是否一致: True


## 8. reshape 四兄弟：pivot_table / melt / stack / explode，以及 merge_asof

四个整理形态的工具经常被孤立地学，放在一起对比才知道该用哪个：

| 方法 | 做什么 | 逆操作 |
|---|---|---|
| `pivot_table` | 长表→宽表，附带聚合 | `melt` |
| `melt` | 宽表→长表（"融"成变量名+值两列） | `pivot`/`pivot_table` |
| `unstack` | 行索引某层→列（第3节已讲） | `stack` |
| `explode` | 一格里的list拆成多行 | `groupby+agg(list)` |

**`merge_asof`**：`merge`只能精确匹配，`merge_asof`按时间做"最近匹配"（比如给每笔订单匹配当天或之前最近一次的汇率），是`merge`处理精确匹配之外的重要补充，金融/时序数据分析常见。

来源：pandas官方`melt`/`merge_asof`文档；Medium "melt vs stack vs pivot vs explode"对比文章。

In [11]:
# melt：把pivot_table生成的宽表"融"回长表，为画图/建模准备
melted = wide.reset_index().melt(id_vars='城市', var_name='月份', value_name='金额')
print("melt后:")
print(melted.head())

# explode：标签列存的是逗号分隔字符串，先split成list，再拆成多行
tags = pd.DataFrame({
    '订单号': [1, 2, 3],
    '标签': ['数码,新品,爆款', '服饰,清仓', '食品']
})
tags['标签'] = tags['标签'].str.split(',')
exploded = tags.explode('标签')
print("\nexplode后每个标签单独一行:")
print(exploded)

melt后:
   城市 月份      金额
0  上海  1  4894.0
1  北京  1  6258.0
2  广州  1  7691.0
3  成都  1  3233.0
4  昆明  1  1618.0

explode后每个标签单独一行:
   订单号  标签
0    1  数码
0    1  新品
0    1  爆款
1    2  服饰
1    2  清仓
2    3  食品


In [12]:
# merge_asof：给每笔交易匹配"当天或之前最近一次"的汇率，不要求时间完全相等
trades = pd.DataFrame({
    '时间': pd.to_datetime(['2024-01-01 09:15', '2024-01-01 14:30', '2024-01-02 10:00']),
    '金额USD': [100, 200, 150]
})
rates = pd.DataFrame({
    '时间': pd.to_datetime(['2024-01-01 08:00', '2024-01-02 08:00']),
    '汇率': [7.10, 7.12]
})
merged = pd.merge_asof(trades, rates, on='时间')   # 按时间"最近匹配"而非精确相等
print(merged)

                   时间  金额USD    汇率
0 2024-01-01 09:15:00    100  7.10
1 2024-01-01 14:30:00    200  7.10
2 2024-01-02 10:00:00    150  7.12


## 9. 本课小结

| 主题 | 一句话记住 |
|---|---|
| 链式+pipe | 消除无语义中间变量，自定义函数用pipe接进去 |
| query/eval | 大表才有性能优势，小表别迷信 |
| MultiIndex | 不是新结构，标签从值变元组；pivot_table=groupby+unstack |
| resample/rolling/expanding | 分别对应"聚合/transform/累计" |
| categorical | 低基数字符串列，省内存不是玄学 |
| CoW | 每次索引都是copy，链式赋值的前提被废除，`.loc`一步赋值永远对 |
| pd.NA / Int64 | 整数带缺失不再被迫变float |
| named agg / transform | 能不用apply就不用，实测有性能差距 |
| melt/explode/merge_asof | 长宽转换 + 拆列表 + 时间最近匹配 |

**下一步**：配套的进阶练习卷 `07_Pandas进阶练习卷.ipynb`，17道题覆盖以上全部知识点，场景化包装（不是干巴巴的语法题）。